## Summary

We implemented two Bayesian Networks using the Expectation-Maximization (EM) algorithm for fraud detection. The small network used three features while the larger network incorporated additional variables. Both models used EM to handle latent variables and estimate conditional probability distributions for predicting fraudulent transactions.

## Methodology

### Small Bayesian Network

**Data Preprocessing:**
- Features: `fraud_bool`, `customer_age`, `income`, `credit_risk_score`
- Discretized continuous variables into 3 bins each (e.g., young/middle-aged/old for age)

**Model Structure:**
- Parent node: `fraud_bool`
- Child nodes: `age_bin`, `income_bin`, `credit_bin`
- Added latent variable `bank_bin` (3 states, never observed)

**Training & Evaluation:**
- Trained using ExpectationMaximization estimator
- Used Variable Elimination for inference on test data
- Evaluated with ROC-AUC score and classification report

### Larger Bayesian Network

**Data Preprocessing:**
- Same binning approach for other variables

**Training & Evaluation:**
- Same EM training approach with latent variable specification
- Evaluated using identical metrics to compare performance with small network

**Key Difference:** The larger network includes more features to potentially capture richer fraud patterns.

In [3]:
!pip install pgmpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 35.0 MB/s eta 0:00:00


**Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pgmpy.estimators import MaximumLikelihoodEstimator, ExpectationMaximization
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/150A datasets/our 10 selected features/balanced_train.csv')


In [ ]:
# Using discrete columns now instead of continous values
bn_df = df[['fraud_bool', 'age_bin', 'income_bin', 'credit_bin']].astype(str)

bn_df

,fraud_bool,age_bin,income_bin,credit_bin
0,0,young,lower class,high credit score
1,0,young,lower class,low credit score
2,0,young,middle class,low credit score
3,0,young,lower class,low credit score
4,1,young,lower class,high credit score
...,...,...,...,...
52934,0,young,upper class,high credit score
52935,0,middle-aged,middle class,low credit score
52936,0,middle-aged,lower class,mid credit score
52937,0,young,middle class,mid credit score


In [ ]:
# defining the BN relationship
model = DiscreteBayesianNetwork([('fraud_bool', 'age_bin'),('fraud_bool', 'income_bin'),('fraud_bool', 'credit_bin'), ('fraud_bool', 'bank_bin')])
# making bank_bin a hidden/latent variable so that it is not used for EM
model.latents.add('bank_bin')
model.fit(bn_df, estimator=ExpectationMaximization, latent_card={'bank_bin': 3})

for cpt in model.get_cpds():
    print(cpt)

  0%|          | 0/100 [00:00<?, ?it/s]

+---------------+----------+
| fraud_bool(0) | 0.833336 |
+---------------+----------+
| fraud_bool(1) | 0.166664 |
+---------------+----------+
+--------------------------+---------------------+---------------------+
| fraud_bool               | fraud_bool(0)       | fraud_bool(1)       |
+--------------------------+---------------------+---------------------+
| income_bin(lower class)  | 0.3623401940339106  | 0.2044656012694095  |
+--------------------------+---------------------+---------------------+
| income_bin(middle class) | 0.42193308550185876 | 0.3599682647625524  |
+--------------------------+---------------------+---------------------+
| income_bin(upper class)  | 0.21572672046423066 | 0.43556613396803806 |
+--------------------------+---------------------+---------------------+
+----------------------+---------------------+---------------------+
| fraud_bool           | fraud_bool(0)       | fraud_bool(1)       |
+----------------------+---------------------+--------------

In [ ]:
# using the testing dataset now
df2 = pd.read_csv('/content/drive/MyDrive/150A datasets/our 10 selected features/balanced_test.csv')

X_test_transformed = df2[['fraud_bool', 'age_bin', 'income_bin', 'credit_bin']].astype(str)

# getting the true fraud values
y_test = X_test_transformed['fraud_bool'].astype(int)

# Inference using built-in library function
inference = VariableElimination(model)

y_prob_em = [] # list for predicted probabilities of fraud
y_pred_em = [] # either 0 or 1

for _, row in X_test_transformed.iterrows():
    evidence = {'age_bin': row['age_bin'], 'income_bin': row['income_bin'], 'credit_bin': row['credit_bin']}
    result = inference.query(variables=['fraud_bool'], evidence=evidence)
    prob = result.values[1]
    y_prob_em.append(prob)
    if prob > 0.5:
        y_pred_em.append(1)
    else:
        y_pred_em.append(0)


print('\nModel Evaluation of EM')
print(f'Classification Report:\n{classification_report(y_test, y_pred_em)}')


Model Evaluation of EM
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91     11029
           1       0.54      0.12      0.20      2206

    accuracy                           0.84     13235
   macro avg       0.69      0.55      0.56     13235
weighted avg       0.80      0.84      0.79     13235



# Larger Baysian Network

In [ ]:
big_df = pd.read_csv('/content/drive/MyDrive/150A datasets/our 10 selected features/balanced_train.csv')
big_df_test= pd.read_csv('/content/drive/MyDrive/150A datasets/our 10 selected features/balanced_test.csv')

In [ ]:
big_df

,fraud_bool,customer_age,name_email_similarity,income,email_is_free,bank_months_count,payment_type,keep_alive_session,has_other_cards,velocity_24h,credit_risk_score
0,0,30,0.320034,0.3,0,-1,2,0,1,4654.102710,202
1,0,20,0.156357,0.2,0,15,1,1,1,4181.158461,36
2,0,10,0.365924,0.5,0,-1,2,0,0,6053.582812,49
3,0,30,0.758116,0.4,1,1,0,1,0,4194.135820,46
4,1,30,0.025710,0.1,1,15,1,1,0,4094.410353,280
...,...,...,...,...,...,...,...,...,...,...,...
52934,0,30,0.879008,0.9,0,-1,2,1,1,5123.788798,342
52935,0,40,0.159300,0.8,1,28,1,1,0,3632.646724,75
52936,0,40,0.835903,0.2,1,16,1,1,0,3288.816690,109
52937,0,30,0.084266,0.8,1,28,0,0,0,5634.782089,116


In [ ]:
bn_df = big_df[['fraud_bool', 'age_bin', 'income_bin', 'credit_bin', 'bank_bin', 'name_email_bin', 'payment_type', 'keep_alive_session', 'has_other_cards', 'velocity_24h_bin', 'email_is_free']].astype(str)


bn_df

,fraud_bool,age_bin,income_bin,credit_bin,bank_bin,name_email_bin,payment_type,keep_alive_session,has_other_cards,velocity_24h_bin,email_is_free
0,0,young,lower class,high credit score,short term,medium similarity,2,0,1,medium,0
1,0,young,lower class,low credit score,medium term,low similar,1,1,1,medium,0
2,0,young,middle class,low credit score,short term,medium similarity,2,0,0,high,0
3,0,young,lower class,low credit score,short term,high similarity,0,1,0,medium,1
4,1,young,lower class,high credit score,medium term,low similar,1,1,0,medium,1
...,...,...,...,...,...,...,...,...,...,...,...
52934,0,young,upper class,high credit score,short term,high similarity,2,1,1,medium,0
52935,0,middle-aged,middle class,low credit score,long term,low similar,1,1,0,low,1
52936,0,middle-aged,lower class,mid credit score,medium term,high similarity,1,1,0,low,1
52937,0,young,middle class,mid credit score,long term,low similar,0,0,0,high,1


In [ ]:
model = DiscreteBayesianNetwork([('fraud_bool', 'age_bin'),('fraud_bool', 'income_bin'),('fraud_bool', 'credit_bin'), ('fraud_bool', 'bank_bin'),
                                 ('fraud_bool', 'name_email_bin'),('fraud_bool', 'payment_type'),('fraud_bool', 'keep_alive_session'),
                                 ('fraud_bool', 'has_other_cards'),('fraud_bool', 'velocity_24h_bin'),('fraud_bool', 'email_is_free')])
model.latents.add('bank_bin')
model.fit(bn_df, estimator=ExpectationMaximization, latent_card={'bank_bin': 3})


for cpt in model.get_cpds():
    print(cpt)

  0%|          | 0/100 [00:00<?, ?it/s]

+--------------------+---------------------+---------------------+
| fraud_bool         | fraud_bool(0)       | fraud_bool(1)       |
+--------------------+---------------------+---------------------+
| has_other_cards(0) | 0.7759996373197933  | 0.9144281990252748  |
+--------------------+---------------------+---------------------+
| has_other_cards(1) | 0.22400036268020673 | 0.08557180097472515 |
+--------------------+---------------------+---------------------+
+-------------------------------+---------------------+---------------------+
| fraud_bool                    | fraud_bool(0)       | fraud_bool(1)       |
+-------------------------------+---------------------+---------------------+
| credit_bin(high credit score) | 0.3314897089491341  | 0.5900487362575088  |
+-------------------------------+---------------------+---------------------+
| credit_bin(low credit score)  | 0.33450448816755823 | 0.19981865578601382 |
+-------------------------------+---------------------+--------

In [ ]:
target = 'fraud_bool'

X_train = big_df.drop(target, axis=1).astype(str)
y_train = big_df[target]
X_test = big_df_test.drop(target, axis=1).astype(str)
y_test = big_df_test[target]

inference = VariableElimination(model)

y_prob_em = [] # list for predicted probabilities of fraud
y_pred_em = [] # either 0 or 1

for _, row in X_test.iterrows():
    evidence = {'age_bin': row['age_bin'], 'income_bin': row['income_bin'], 'credit_bin': row['credit_bin'],
                'name_email_bin': row['name_email_bin'], 'payment_type': row['payment_type'],'keep_alive_session': row['keep_alive_session'],
                'velocity_24h_bin': row['velocity_24h_bin'],'email_is_free': row['email_is_free']}
    result = inference.query(variables=['fraud_bool'], evidence=evidence)
    prob = result.values[1]
    y_prob_em.append(prob)
    if prob > 0.5:
        y_pred_em.append(1)
    else:
        y_pred_em.append(0)


print('\nModel Evaluation of EM')
print(f'Classification Report:\n{classification_report(y_test, y_pred_em)}')



Model Evaluation of EM
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91     11029
           1       0.59      0.27      0.37      2206

    accuracy                           0.85     13235
   macro avg       0.73      0.62      0.64     13235
weighted avg       0.82      0.85      0.82     13235



# Small Bayesian Network (new balanced data)

In [5]:
df_2 = pd.read_csv('/balanced_train.csv')

In [7]:
# Using discrete columns now instead of continous values
bn_df2 = df_2[['fraud_bool', 'age_bin', 'income_bin', 'credit_bin']].astype(str)

bn_df2

,fraud_bool,age_bin,income_bin,credit_bin
0,0,young,lower class,high credit
1,0,young,middle class,very low credit
2,0,middle-aged,middle class,high credit
3,0,young,upper class,low credit
4,1,young,lower class,very high credit
...,...,...,...,...
52934,0,middle-aged,middle class,very high credit
52935,0,middle-aged,lower class,medium credit
52936,0,middle-aged,middle class,very high credit
52937,0,old,lower class,very high credit


In [8]:
# defining the BN relationship
model2 = DiscreteBayesianNetwork([('fraud_bool', 'age_bin'),('fraud_bool', 'income_bin'),('fraud_bool', 'credit_bin'), ('fraud_bool', 'bank_bin')])
# making bank_bin a hidden/latent variable so that it is not used for EM
model2.latents.add('bank_bin')
model2.fit(bn_df2, estimator=ExpectationMaximization, latent_card={'bank_bin': 3})

for cpt in model2.get_cpds():
    print(cpt)

  0%|          | 0/100 [00:00<?, ?it/s]

+---------------+----------+
| fraud_bool(0) | 0.833336 |
+---------------+----------+
| fraud_bool(1) | 0.166664 |
+---------------+----------+
+--------------------------+---------------------+---------------------+
| fraud_bool               | fraud_bool(0)       | fraud_bool(1)       |
+--------------------------+---------------------+---------------------+
| income_bin(lower class)  | 0.36435760268383355 | 0.2044656012694095  |
+--------------------------+---------------------+---------------------+
| income_bin(middle class) | 0.4155181793453622  | 0.3599682647625524  |
+--------------------------+---------------------+---------------------+
| income_bin(upper class)  | 0.22012421797080425 | 0.43556613396803806 |
+--------------------------+---------------------+---------------------+
+------------------------------+---------------------+---------------------+
| fraud_bool                   | fraud_bool(0)       | fraud_bool(1)       |
+------------------------------+------------

In [10]:
# using the testing dataset now
df_2_test = pd.read_csv('/balanced_test.csv')

X_test_transformed = df_2_test[['fraud_bool', 'age_bin', 'income_bin', 'credit_bin']].astype(str)

# getting the true fraud values
y_test = X_test_transformed['fraud_bool'].astype(int)

# Inference using built-in library function
inference = VariableElimination(model2)

y_prob_em = [] # list for predicted probabilities of fraud
y_pred_em = [] # either 0 or 1

for _, row in X_test_transformed.iterrows():
    evidence = {'age_bin': row['age_bin'], 'income_bin': row['income_bin'], 'credit_bin': row['credit_bin']}
    result = inference.query(variables=['fraud_bool'], evidence=evidence)
    prob = result.values[1]
    y_prob_em.append(prob)
    if prob > 0.5:
        y_pred_em.append(1)
    else:
        y_pred_em.append(0)


print('\nModel Evaluation of EM')
print(f'Classification Report:\n{classification_report(y_test, y_pred_em)}')


Model Evaluation of EM
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.99      0.91     11029
           1       0.57      0.10      0.17      2206

    accuracy                           0.84     13235
   macro avg       0.71      0.54      0.54     13235
weighted avg       0.80      0.84      0.79     13235



# Large Baysian Network (new balanced data)

In [11]:
big_df2 = pd.read_csv('/balanced_train.csv')
big_df2_test= pd.read_csv('/balanced_test.csv')

In [12]:
big_df2

,fraud_bool,email_is_free,payment_type,keep_alive_session,has_other_cards,age_bin,income_bin,credit_bin,bank_bin,name_email_bin,velocity_24h_bin
0,0,0,2,1,0,young,lower class,high credit,short term,low similarity,very low
1,0,1,1,0,0,young,middle class,very low credit,medium term,very low similarity,low
2,0,1,3,1,0,middle-aged,middle class,high credit,short term,very low similarity,high
3,0,1,1,1,0,young,upper class,low credit,short term,high similarity,low
4,1,1,1,1,0,young,lower class,very high credit,medium term,very low similarity,low
...,...,...,...,...,...,...,...,...,...,...,...
52934,0,0,0,1,0,middle-aged,middle class,very high credit,long term,very high similarity,medium
52935,0,1,0,0,1,middle-aged,lower class,medium credit,short term,very high similarity,low
52936,0,0,0,0,0,middle-aged,middle class,very high credit,long term,very low similarity,high
52937,0,1,1,1,1,old,lower class,very high credit,short term,very low similarity,low


In [16]:
bn_df2 = big_df2[['fraud_bool', 'age_bin', 'income_bin', 'credit_bin', 'bank_bin', 'name_email_bin', 'payment_type', 'keep_alive_session', 'has_other_cards', 'velocity_24h_bin', 'email_is_free']].astype(str)


bn_df2

,fraud_bool,age_bin,income_bin,credit_bin,bank_bin,name_email_bin,payment_type,keep_alive_session,has_other_cards,velocity_24h_bin,email_is_free
0,0,young,lower class,high credit,short term,low similarity,2,1,0,very low,0
1,0,young,middle class,very low credit,medium term,very low similarity,1,0,0,low,1
2,0,middle-aged,middle class,high credit,short term,very low similarity,3,1,0,high,1
3,0,young,upper class,low credit,short term,high similarity,1,1,0,low,1
4,1,young,lower class,very high credit,medium term,very low similarity,1,1,0,low,1
...,...,...,...,...,...,...,...,...,...,...,...
52934,0,middle-aged,middle class,very high credit,long term,very high similarity,0,1,0,medium,0
52935,0,middle-aged,lower class,medium credit,short term,very high similarity,0,0,1,low,1
52936,0,middle-aged,middle class,very high credit,long term,very low similarity,0,0,0,high,0
52937,0,old,lower class,very high credit,short term,very low similarity,1,1,1,low,1


In [17]:
model2 = DiscreteBayesianNetwork([('fraud_bool', 'age_bin'),('fraud_bool', 'income_bin'),('fraud_bool', 'credit_bin'), ('fraud_bool', 'bank_bin'),
                                 ('fraud_bool', 'name_email_bin'),('fraud_bool', 'payment_type'),('fraud_bool', 'keep_alive_session'),
                                 ('fraud_bool', 'has_other_cards'),('fraud_bool', 'velocity_24h_bin'),('fraud_bool', 'email_is_free')])
model2.latents.add('bank_bin')
model2.fit(bn_df2, estimator=ExpectationMaximization, latent_card={'bank_bin': 3})


for cpt in model2.get_cpds():
    print(cpt)

  0%|          | 0/100 [00:00<?, ?it/s]

+--------------------+-------------------+---------------------+
| fraud_bool         | fraud_bool(0)     | fraud_bool(1)       |
+--------------------+-------------------+---------------------+
| has_other_cards(0) | 0.775704959651827 | 0.9144281990252748  |
+--------------------+-------------------+---------------------+
| has_other_cards(1) | 0.224295040348173 | 0.08557180097472515 |
+--------------------+-------------------+---------------------+
+----------------------+---------------------+---------------------+
| fraud_bool           | fraud_bool(0)       | fraud_bool(1)       |
+----------------------+---------------------+---------------------+
| age_bin(middle-aged) | 0.23977695167286245 | 0.25966224640145075 |
+----------------------+---------------------+---------------------+
| age_bin(old)         | 0.17857466678756007 | 0.3877365975291851  |
+----------------------+---------------------+---------------------+
| age_bin(young)       | 0.5816483815395774  | 0.3526011560693

In [19]:
target = 'fraud_bool'

X_train = big_df2.drop(target, axis=1).astype(str)
y_train = big_df2[target]
X_test = big_df2_test.drop(target, axis=1).astype(str)
y_test = big_df2_test[target]

inference = VariableElimination(model2)

y_prob_em = [] # list for predicted probabilities of fraud
y_pred_em = [] # either 0 or 1

for _, row in X_test.iterrows():
    evidence = {'age_bin': row['age_bin'], 'income_bin': row['income_bin'], 'credit_bin': row['credit_bin'],
                'name_email_bin': row['name_email_bin'], 'payment_type': row['payment_type'],'keep_alive_session': row['keep_alive_session'],
                'velocity_24h_bin': row['velocity_24h_bin'],'email_is_free': row['email_is_free']}
    result = inference.query(variables=['fraud_bool'], evidence=evidence)
    prob = result.values[1]
    y_prob_em.append(prob)
    if prob > 0.5:
        y_pred_em.append(1)
    else:
        y_pred_em.append(0)


print('\nModel Evaluation of EM with new balanced data')
print(f'Classification Report:\n{classification_report(y_test, y_pred_em)}')



Model Evaluation of EM with new balanced data
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91     11029
           1       0.57      0.30      0.39      2206

    accuracy                           0.85     13235
   macro avg       0.72      0.63      0.65     13235
weighted avg       0.82      0.85      0.82     13235

